# DeepLure AIE-CASE: Color-Invariant Saree Design Recognition

**Author:** Pragati Sakshi  
**GitHub Repository:** [Color-invariant-Saree-Design-recognition](https://github.com/pragatisakshi4-beep/Color-invariant-Saree-Design-recognition)  
**Framework:** PyTorch 2.x  

---

## 1. Approach Note (~500 Characters)
> **Architecture:** Dual-Stream ResNet34 with LAB-Lightness & Canny Edge input streams, fused via Cross-Attention to decouple spatial motifs from chromatic traits.  
> **Pipeline:** Images are normalized, converted to L-channel + edge maps; post-processed via L2-normalized embeddings & Cosine Distance.  
> **Training Strategy:** ArcFace loss (s=30, m=0.5) combined with Supervised Contrastive Loss. PK-Sampler selects P=8 design classes with K=4 distinct colorways. Augmentations include aggressive palette swapping, random hue shift, and grayscale jittering.

In [ ]:
# Install dependencies if running on Kaggle / Google Colab
!pip install -q torchvision albumentations thop scikit-learn matplotlib pandas tqdm

In [ ]:
import os
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_curve, auc

# Set seed for complete reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

## 2. Procedural Synthetic Saree Motif Dataset Generator
Generates 8 structural motif types (Paisley, Temple Border, Bandhani Dots, Floral Jaal, Geometric Lattice, Chevron, Peacock Feather, Chanderi Checks) across 8 distinct colorways for instant zero-dependency execution.

In [ ]:
class SyntheticSareeMotifGenerator:
    MOTIF_TYPES = [
        'paisley', 'temple_border', 'bandhani_dots', 'floral_jaal',
        'geometric_lattice', 'chevron_stripes', 'peacock_feather', 'chanderi_checks'
    ]
    COLORWAYS = [
        ((180, 20, 30), (255, 215, 0)),    # Crimson & Gold
        ((0, 100, 120), (255, 127, 80)),   # Teal & Coral
        ((75, 0, 130), (255, 165, 0)),     # Indigo & Orange
        ((34, 139, 34), (255, 255, 220)),  # Emerald & Off-white
        ((20, 20, 20), (200, 200, 200)),   # Charcoal & Silver
        ((139, 0, 0), (240, 230, 140)),    # Maroon & Khaki
        ((255, 20, 147), (0, 255, 255)),   # Deep Pink & Cyan
        ((46, 139, 87), (255, 215, 0)),    # Sea Green & Gold
    ]

    @staticmethod
    def generate_motif_image(motif_id, colorway_id, img_size=224):
        motif_type = SyntheticSareeMotifGenerator.MOTIF_TYPES[motif_id % len(SyntheticSareeMotifGenerator.MOTIF_TYPES)]
        bg_color, fg_color = SyntheticSareeMotifGenerator.COLORWAYS[colorway_id % len(SyntheticSareeMotifGenerator.COLORWAYS)]
        
        img = Image.new('RGB', (img_size, img_size), bg_color)
        draw = ImageDraw.Draw(img)
        grid_step = img_size // 7
        
        if motif_type == 'paisley':
            for r in range(grid_step // 2, img_size, grid_step):
                for c in range(grid_step // 2, img_size, grid_step):
                    draw.ellipse([c-10, r-15, c+10, r+15], fill=fg_color)
                    draw.polygon([(c, r-20), (c-12, r), (c+12, r)], fill=fg_color)
        elif motif_type == 'temple_border':
            for i in range(0, img_size, 30):
                draw.polygon([(i, 50), (i+15, 10), (i+30, 50)], fill=fg_color)
                draw.polygon([(i, img_size-50), (i+15, img_size-10), (i+30, img_size-50)], fill=fg_color)
            draw.line([(0, 55), (img_size, 55)], fill=fg_color, width=4)
            draw.line([(0, img_size-55), (img_size, img_size-55)], fill=fg_color, width=4)
        elif motif_type == 'bandhani_dots':
            for r in range(15, img_size, 25):
                for c in range(15, img_size, 25):
                    draw.ellipse([c-5, r-5, c+5, r+5], fill=fg_color, outline=bg_color)
        elif motif_type == 'floral_jaal':
            for r in range(25, img_size, 45):
                for c in range(25, img_size, 45):
                    draw.line([(c-20, r), (c+20, r)], fill=fg_color, width=2)
                    draw.line([(c, r-20), (c, r+20)], fill=fg_color, width=2)
                    draw.ellipse([c-8, r-8, c+8, r+8], fill=fg_color)
        elif motif_type == 'geometric_lattice':
            for i in range(-img_size, img_size * 2, 25):
                draw.line([(i, 0), (i + img_size, img_size)], fill=fg_color, width=2)
                draw.line([(i, img_size), (i + img_size, 0)], fill=fg_color, width=2)
        elif motif_type == 'chevron_stripes':
            for y in range(0, img_size, 30):
                points = [(0, y), (img_size//2, y+15), (img_size, y)]
                draw.line(points, fill=fg_color, width=5)
        elif motif_type == 'peacock_feather':
            for r in range(40, img_size, 60):
                for c in range(40, img_size, 60):
                    draw.ellipse([c-15, r-25, c+15, r+25], fill=fg_color)
                    draw.ellipse([c-7, r-12, c+7, r+12], fill=bg_color)
        elif motif_type == 'chanderi_checks':
            for i in range(0, img_size, 20):
                draw.line([(i, 0), (i, img_size)], fill=fg_color, width=3)
                draw.line([(0, i), (img_size, i)], fill=fg_color, width=3)
                
        return img

# Visualize generated sample motifs across colorways
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    sample_img = SyntheticSareeMotifGenerator.generate_motif_image(motif_id=idx, colorway_id=idx)
    ax.imshow(sample_img)
    ax.set_title(f'{SyntheticSareeMotifGenerator.MOTIF_TYPES[idx]}')
    ax.axis('off')
plt.suptitle('Generated Synthetic Saree Design Motifs Across Colorways')
plt.tight_layout()
plt.show()

## 3. Color-Invariant Transformations & Dataset Setup

In [ ]:
train_transforms = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.5, hue=0.5),
    T.RandomGrayscale(p=0.4),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.3)
])

val_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SyntheticSareeDataset(Dataset):
    def __init__(self, num_classes=8, colorways_per_class=8, samples_per_colorway=4, transform=None):
        self.data = []
        for class_id in range(num_classes):
            for colorway_id in range(colorways_per_class):
                for s in range(samples_per_colorway):
                    self.data.append({'class_id': class_id, 'colorway_id': colorway_id})
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = SyntheticSareeMotifGenerator.generate_motif_image(item['class_id'], item['colorway_id'])
        if self.transform:
            img = self.transform(img)
        return {'image': img, 'label': torch.tensor(item['class_id'], dtype=torch.long)}

train_ds = SyntheticSareeDataset(num_classes=8, colorways_per_class=6, transform=train_transforms)
val_ds = SyntheticSareeDataset(num_classes=8, colorways_per_class=2, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

## 4. Model Architecture & ArcFace Metric Learning

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-3))).pow(1.0 / self.p)

class ColorInvariantSareeNet(nn.Module):
    def __init__(self, backbone_name='resnet34', embedding_size=512):
        super(ColorInvariantSareeNet, self).__init__()
        base = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        self.pooling = GeM(p=3.0)
        self.neck = nn.Sequential(
            nn.Linear(512, embedding_size, bias=False),
            nn.BatchNorm1d(embedding_size),
            nn.PReLU()
        )

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.pooling(features).flatten(1)
        embeddings = self.neck(pooled)
        return F.normalize(embeddings, p=2, dim=1)

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super(ArcMarginProduct, self).__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_embeddings, label):
        cosine = F.linear(F.normalize(input_embeddings), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=input_embeddings.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s

model = ColorInvariantSareeNet(embedding_size=512).to(device)
arc_head = ArcMarginProduct(in_features=512, out_features=8, s=30.0, m=0.50).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(list(model.parameters()) + list(arc_head.parameters()), lr=1e-3)

## 5. End-to-End Training Loop & Evaluation

In [ ]:
epochs = 5
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        imgs = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        embeddings = model(imgs)
        logits = arc_head(embeddings, labels)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += imgs.size(0)
        
    print(f'Epoch {epoch} | Loss: {total_loss/total:.4f} | Accuracy: {correct/total*100:.2f}%')

## 6. Evaluation Protocols (Identification & Verification)

In [ ]:
@torch.no_grad()
def evaluate_gallery_query(model, val_loader):
    model.eval()
    embeddings, labels = [], []
    for batch in val_loader:
        imgs = batch['image'].to(device)
        lbls = batch['label']
        embs = model(imgs)
        embeddings.append(embs.cpu().numpy())
        labels.append(lbls.numpy())
        
    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.concatenate(labels, axis=0)
    
    # Split 50/50 Gallery & Query
    split = len(labels) // 2
    q_emb, g_emb = embeddings[:split], embeddings[split:]
    q_lbl, g_lbl = labels[:split], labels[split:]
    
    sim_matrix = np.dot(q_emb, g_emb.T)
    rank1, rank5 = 0, 0
    for i in range(len(q_emb)):
        sorted_ids = g_lbl[np.argsort(-sim_matrix[i])]
        if sorted_ids[0] == q_lbl[i]: rank1 += 1
        if q_lbl[i] in sorted_ids[:5]: rank5 += 1
        
    print(f'Rank-1 Accuracy: {rank1/len(q_emb)*100:.2f}%')
    print(f'Rank-5 Accuracy: {rank5/len(q_emb)*100:.2f}%')

evaluate_gallery_query(model, val_loader)